In [1]:
!pip install SpeechRecognition pyaudio together


In [12]:
!pip install pyttsx3

In [1]:
import os
import pandas as pd
import speech_recognition as sr
import pyttsx3
from together import Together

# Set Together.AI API key
os.environ["TOGETHER_API_KEY"] = "dfd4b5dc8c148f418ea5b2702bd8721a21ca6fabe3e8dc1c511fc7abae24c0a7"

# Initialize Together.AI client
client = Together()

# Load the Excel file
file_path = "candidate_resume.xlsx"
df = pd.read_excel(file_path)

# Extract job description and resume
job_description = df.loc[0, "Job Description"]
resume = df.loc[0, "Resume"]

# Initialize text-to-speech engine
engine = pyttsx3.init()

# Speech recognition setup
recognizer = sr.Recognizer()

# Store Q&A pairs
qa_pairs = []

for i in range(4):
    # Generate interview question
    prompt = f"Based on the following job description and resume, generate a relevant interview question(generate only question without any description of question):\n\nJob Description: {job_description}\n\nResume: {resume}\n\nPrevious Q&A: {qa_pairs}\n\nNext Interview Question:"
    response = client.chat.completions.create(model="meta-llama/Llama-Vision-Free", messages=[{"role": "user", "content": prompt}])
    question = response.choices[0].message.content.strip()
    
    # Speak the question
    print(f"AI: {question}")
    engine.say(question)
    engine.runAndWait()
    
    # Record the candidate's answer
    with sr.Microphone() as source:
        print("Candidate, please answer:")
        recognizer.adjust_for_ambient_noise(source)
        audio = recognizer.listen(source)
    
    try:
        answer = recognizer.recognize_google(audio)
    except sr.UnknownValueError:
        answer = "[Unrecognized speech]"
    
    print(f"Candidate's Answer: {answer}")
    
    # Store Q&A pair
    qa_pairs.append({"Question": question, "Answer": answer})

# Save Q&A pairs to Excel
output_df = pd.DataFrame(qa_pairs)
output_file = "interview_transcript.xlsx"
output_df.to_excel(output_file, index=False)
print(f"Interview transcript saved to {output_file}")


AI: How do you stay current with the latest developments and advancements in software development, particularly in the areas of scalability, performance, and cloud computing?
Candidate, please answer:
Candidate's Answer: [Unrecognized speech]
AI: Can you walk me through your thought process and technical approach when deciding which programming language or framework to use for a given project, and how you balance the trade-offs between development speed, maintainability, and scalability?
Candidate, please answer:
Candidate's Answer: [Unrecognized speech]
AI: Can you walk me through a specific example of how you developed and maintained a software application, and what were some of the challenges you faced and how you overcame them?
Candidate, please answer:
Candidate's Answer: [Unrecognized speech]
AI: Can you explain the benefits and trade-offs of using a microservices architecture versus a monolithic architecture in software application development?
Candidate, please answer:
Candidat

In [4]:
transcript_file = "interview_transcript.xlsx"
resume_file = "candidate_resume.xlsx"
df_transcript = pd.read_excel(transcript_file)
df_resume = pd.read_excel(resume_file)
print(df_transcript.columns)
print(df_resume.columns)


Index(['Question', 'Answer'], dtype='object')
Index(['Name', 'Role', 'Job Description', 'Resume'], dtype='object')


In [6]:
import pandas as pd

# Load datasets
transcript_file = "interview_transcript.xlsx"
resume_file = "candidate_resume.xlsx"

df_transcript = pd.read_excel(transcript_file)
df_resume = pd.read_excel(resume_file)

# Ensure 'Name' column is present in transcript by mapping names from resume
df_transcript["Name"] = df_resume["Name"].iloc[:len(df_transcript)]  # Assuming row order aligns

# Combine "Question" and "Answer" into "Transcript"
df_transcript["Transcript"] = df_transcript.apply(lambda row: f"{row['Question']}\n{row['Answer']}", axis=1)

# Concatenate all Q&A into a single string per candidate
df_transcript = df_transcript.groupby("Name")["Transcript"].apply(lambda x: "\n".join(x)).reset_index()

# Save the updated file (overwrite original)
df_transcript.to_excel(transcript_file, index=False)
print("interview_transcript.xlsx updated successfully with 'Name' and 'Transcript' columns!")


interview_transcript.xlsx updated successfully with 'Name' and 'Transcript' columns!


In [11]:
import pandas as pd
import joblib
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
resume_file = "candidate_resume.xlsx"
transcript_file = "interview_transcript.xlsx"
model_file = "xgb_model.pkl"

df_resume = pd.read_excel(resume_file)
df_transcript = pd.read_excel(transcript_file)

# Merge datasets on 'Name'
df_merged = pd.merge(df_resume, df_transcript, on="Name", how="left")
df_merged["Transcript"] = df_merged["Transcript"].fillna("")

# Text cleaning function
def clean_text(text):
    if pd.notnull(text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return text
    return ""

# Apply text cleaning
df_merged["Cleaned_Resume"] = df_merged["Resume"].apply(clean_text)
df_merged["Cleaned_Transcript"] = df_merged["Transcript"].apply(clean_text)
df_merged["Cleaned_Job_Description"] = df_merged["Job Description"].apply(clean_text)

# Feature extraction functions
def count_words(text):
    return len(text.split()) if pd.notnull(text) else 0

def avg_word_length(text):
    words = text.split() if pd.notnull(text) else []
    return sum(len(word) for word in words) / len(words) if words else 0

def unique_word_ratio(text):
    words = text.split() if pd.notnull(text) else []
    return len(set(words)) / len(words) if words else 0

def keyword_count(text, keywords):
    words = text.split() if pd.notnull(text) else []
    return sum(1 for word in words if word.lower() in keywords)

def keyword_overlap(text1, text2):
    if pd.notnull(text1) and pd.notnull(text2):
        words1 = set(text1.split())
        words2 = set(text2.split())
        return len(words1 & words2)
    return 0

# Predefined keyword dictionaries
technical_keywords = {'python', 'java', 'sql', 'machine learning', 'cloud', 'design', 'analysis', 'management'}
positive_keywords = {'excellent', 'success', 'outstanding', 'achievement', 'skilled'}
negative_keywords = {'poor', 'inadequate', 'lacking', 'failure', 'weak'}

# Apply feature extraction
df_merged['resume_positive_keyword_count'] = df_merged['Cleaned_Resume'].apply(lambda x: keyword_count(x, positive_keywords))
df_merged['resume_negative_keyword_count'] = df_merged['Cleaned_Resume'].apply(lambda x: keyword_count(x, negative_keywords))
df_merged['resume_char_count'] = df_merged['Cleaned_Resume'].apply(len)
df_merged['resume_job_keyword_overlap'] = df_merged.apply(lambda row: keyword_overlap(row['Cleaned_Resume'], row['Cleaned_Job_Description']), axis=1)

df_merged['transcript_positive_keyword_count'] = df_merged['Cleaned_Transcript'].apply(lambda x: keyword_count(x, positive_keywords))
df_merged['transcript_char_count'] = df_merged['Cleaned_Transcript'].apply(len)
df_merged['transcript_avg_word_length'] = df_merged['Cleaned_Transcript'].apply(avg_word_length)
df_merged['transcript_unique_word_ratio'] = df_merged['Cleaned_Transcript'].apply(unique_word_ratio)
df_merged['transcript_job_keyword_overlap'] = df_merged.apply(lambda row: keyword_overlap(row['Cleaned_Transcript'], row['Cleaned_Job_Description']), axis=1)

# Feature selection
features = [
    'transcript_positive_keyword_count',
    'resume_positive_keyword_count',
    'transcript_avg_word_length',
    'transcript_char_count',
    'transcript_job_keyword_overlap',
    'resume_negative_keyword_count',
    'resume_job_keyword_overlap',
    'resume_char_count',
    'transcript_unique_word_ratio'
]

# Load the trained model and make predictions
xgb_model = joblib.load(model_file)
df_merged['Selection_Prediction'] = xgb_model.predict(df_merged[features])

# Save processed data
df_merged.to_excel("processed_results.xlsx", index=False)
print("Processed data with predictions saved to processed_results.xlsx")



Processed data with predictions saved to processed_results.xlsx


C:\Users\Hari Martha\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [14:21:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\data\../common/error_msg.h:80: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  warnings.warn(smsg, UserWarning)


In [13]:
import pandas as pd

# Load dataset
processed_results_file = "processed_results.xlsx"
df_results = pd.read_excel(processed_results_file)

# Apply selection logic
df_results["Selection_Status"] = df_results["Selection_Prediction"].apply(lambda x: "Selected" if x >= 0.5 else "Not Selected")

# Print selection status
print(df_results[["Name", "Selection_Status"]])

       Name Selection_Status
0  John Doe     Not Selected


In [14]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

def send_email(name, selection_status):
    from_email = "2713alpha8631@gmail.com"
    from_password = "pldf ttue xzte decz"  # Update this line with your app password
    to_email = "6520mhari8631@gmail.com"
    
    msg = MIMEMultipart()
    msg['From'] = from_email
    msg['To'] = to_email
    msg['Subject'] = f"Selection Status for {name}"
    
    body = f"Candidate {name} has been {selection_status}."
    msg.attach(MIMEText(body, 'plain'))
    
    with smtplib.SMTP('smtp.gmail.com', 587) as server:
        server.starttls()
        server.login(from_email, from_password)
        server.sendmail(from_email, to_email, msg.as_string())

# Send emails for each candidate
for _, row in df_results.iterrows():
    send_email(row["Name"], row["Selection_Status"])
